# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example for loading and exploring the FAIR² dataset package using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Ignore SettingWithCopy warnings for cleaner output
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Examine the available record sets (tables) and fields in the dataset. All entities are referenced by their `@id` fields.

In [ ]:
# List record sets present in the dataset
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Available record sets:")
    for rs in metadata.recordSet:
        print(f"  - Record Set @id: {rs['@id']}, name: {rs.get('name', '(unnamed)')}")
else:
    # mlcroissant may provide a get_record_set_ids method
    print("Retrieving record sets via dataset API...")
    record_sets = dataset.record_sets()
    for rs in record_sets:
        print(f"  - Record Set @id: {rs['@id']}, name: {rs.get('name', '(unnamed)')}")
    if not record_sets:
        print("Warning: No record sets found. The dataset might not be cataloged with explicit record sets in the Croissant schema.")

In [ ]:
# Show fields (columns) for each record set by their @id
record_sets = dataset.record_sets()
print("\nFields available in each record set:")
for record_set in record_sets:
    print(f"\nRecord set: {record_set['@id']} (name: {record_set.get('name', '(unnamed)')})")
    if 'field' in record_set:
        fields = record_set['field']
        # If only one field, 'field' is a dict, else list
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  - Field @id: {field['@id']}, name: {field.get('name', '(unnamed)')}")
    else:
        print("  (No fields found in this record set)")

### Example: Print a few sample records from a record set
You can replace the sample_record_set_id below with the `@id` of a real record set from the output above.

In [ ]:
# Find the first available record set @id to use for examples
record_sets = dataset.record_sets()
if record_sets:
    sample_record_set_id = record_sets[0]['@id']
    print(f"Showing a few example records for record set @id: {sample_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if i >= 2:  # Show only first 3 records
            break
else:
    print("No record sets found!")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.
Use the record set and field `@id`s from the previous overview. All entities are referenced by their `@id`.

In [ ]:
# Gather all record set @ids
record_sets = dataset.record_sets()
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting data from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if there is data
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(dataframes[record_set_id])} records.")
    else:
        print("  (No records found)")

# Choose a main record set to show columns and preview data
main_record_set_id = None
for k, df in dataframes.items():
    if not df.empty:
        main_record_set_id = k
        break

if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"\nColumns in record set {main_record_set_id}:\n{df.columns.tolist()}")
    print("\nFirst few rows:")
    display(df.head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common analysis steps: filtering, normalization, grouping. For all references, use `@id` for record set and columns.

In [ ]:
# Choose a numeric field @id for demonstration (examine columns above; update as needed)
numeric_field_id = None
group_field_id = None
df = None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to guess a numeric column; fallback if none found
    for col in df.columns:
        # Guess numeric by dtype or column name
        if pd.api.types.is_numeric_dtype(df[col]) or 'age' in col.lower() or 'interval' in col.lower() or 'count' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        if len(df.columns) > 0:
            numeric_field_id = df.columns[0]  # Use the first column as fallback
    # Try to guess a grouping field (like sex, gender, anatomical location, etc.)
    for col in df.columns:
        if 'sex' in col.lower() or 'location' in col.lower() or 'msi' in col.lower():
            group_field_id = col
            break

    print(f"Using numeric field @id: {numeric_field_id}")
    if group_field_id:
        print(f"Using group field @id: {group_field_id}")

    # Filtering example
    threshold = None
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).any() else 0
    else:
        # Try converting to numeric
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id]).any() else 0
        except Exception:
            threshold = 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
            filtered_df[numeric_field_id].std(ddof=0)
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No DataFrame found for analysis.")

## 5. Visualization
Visualize the numeric field distribution and group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we've used the `mlcroissant` library to explore the FAIR² dataset defined via a Croissant schema. By referencing all record sets and fields via their `@id` fields, we loaded data, performed example filtering and normalization, and visualized key distributions.

This workflow provides a reproducible and schema-driven approach for tabular FAIR clinical datasets. Extend these steps for further analysis and modeling as needed for your use-case!